# 06 — Nominatim Geocoder
Second-pass geocoding via the Nominatim API for rows notebook 05's tier-based matcher couldn't resolve.

Pipeline position: OCR -> Evaluation -> Version Selection -> Statistics -> Geocoder -> Nominatim Geocoder


## Imports


In [ ]:
import re
import io
import json
import time
import tempfile
import urllib.request
import urllib.parse
import warnings
from pathlib import Path
from datetime import datetime

import openai
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore")

from config import (
    OPENAI_KEY_FILE,
    GEOCODER_OUTPUT_DIR,
    NOMINATIM_OUTPUT_DIR,      
    DRIVE_ROOT_FOLDER,
    DRIVE_GEOSPATIAL_FOLDER,
    DRIVE_OUTPUTS_FOLDER,
    DRIVE_CLEAN_PAGES_FOLDER,
    DRIVE_NOMINATIM_FOLDER,    
    CLEAN_PAGES_DIR,
    SESTIERE_PATH, 
    FINAL_OUTPUT_DIR,
    DRIVE_FINAL_FOLDER,
    LOCAL_ALMANAC_ROOT,
    SAVE_MODE,
)


## Configuration
RUN_OUTSIDE / RUN_INSIDE toggle which of Cells 4/5 execute.


In [ ]:
# ── Execution flags ──────────────────────────────────────────
# SAVE_MODE now lives in config.py 
RUN_OUTSIDE  = True   
RUN_INSIDE   = True   

print("=== NOTEBOOK 06 — NOMINATIM GEOCODER ===")
print(f"  SAVE_MODE   : {SAVE_MODE}")
print(f"  RUN_OUTSIDE : {RUN_OUTSIDE}")
print(f"  RUN_INSIDE  : {RUN_INSIDE}")

=== NOTEBOOK 06 — NOMINATIM GEOCODER ===
  SAVE_MODE   : both
  RUN_OUTSIDE : True
  RUN_INSIDE  : True

## Reference data
Outside-Venice place regex, town centroids, Nominatim accept/reject rules, section hints for the geocoding prompt.


In [ ]:
# ── Corrected OUTSIDE_VENICE_RE ─────────────────────────────
OUTSIDE_VENICE_RE = re.compile(
    r"\b("
    r"burano|murano|mestre|marghera|porto marghera|lido|"
    r"chioggia|zelarino|favaro veneto|favaro|pellestrina|"
    r"dese|carpenedo|jesolo|spinea|mira|dolo|chirignago|"
    r"tessera|martellago|mirano|salzano|mogliano|"
    r"cavallino|treporti|malamocco|"
    r"s\.\s*pietro in volta|"
    r"annone veneto|campagnalupia|campolongo maggiore|"
    r"camponogara|caorle|cavarzere|ceggia|"
    r"cinto caomaggiore|cona|concordia sagittaria|"
    r"fiess[eo]\s+d.artico|"
    r"fossalta di piave|fossalta di portogruaro|"
    r"foss[oò]'?|grisolera|gruaro|marcon|meolo|"
    r"musile di piave|noale|noventa di piave|pianiga|"
    r"portogruaro|pramaggiore|"
    r"s\.\s*dona'?\s+di\s+piave|san\s+don[aà]\s+di\s+piave|"
    r"s\.\s*maria\s+di\s+sala|santa\s+maria\s+di\s+sala|"
    r"s\.\s*michele\s+al\s+tagliamento|san\s+michele\s+al\s+tagliamento|"
    r"s\.\s*michele\s+del\s+quarto|"
    r"s\.\s*stino\s+di\s+livenza|san\s+stino\s+di\s+livenza|"
    r"teglio veneto|torre di mosto|vigonovo|"
    r"scorz[eè]|"
    r"viale delle industrie"
    r")\b",
    re.IGNORECASE
)
#STREET_PREFIX_RE is the gate between "try to geocode this address" and "just use town centroid".
STREET_PREFIX_RE = re.compile(
    r"\b(via|viale|strada|calle|fondamenta|riva|campo|"
    r"campiello|corte|sinistra|destra)\b",
    re.IGNORECASE
)

CIVIC_NUM_RE = re.compile(r"\b(\d{1,5})\b")

# ── Town centroids (outside Venice) ─────────────────────────
TOWN_CENTROIDS = {
    'alberoni':                    (45.348, 12.3295),
    'annone veneto':               (45.7944, 12.6829),
    'burano':                      (45.4854, 12.4175),
    'campagnalupia':               (45.3484, 12.1647),
    'campolongo maggiore':         (45.3305, 12.0483),
    'camponogara':                 (45.3769, 12.0921),
    'caorle':                      (45.6378, 12.9058),
    'carpenedo':                   (45.5047, 12.2508),
    'cavallino':                   (45.4820, 12.5529),
    'cavarzere':                   (45.1130, 12.1329),
    'ceggia':                      (45.6827, 12.6429),
    'chioggia':                    (45.2179, 12.2271),
    'chirignago':                  (45.4847, 12.1888),
    'cinto caomaggiore':           (45.8220, 12.7818),
    'cona':                        (45.1801, 12.0787),
    'concordia sagittaria':        (45.7561, 12.8450),
    'dese':                        (45.5268, 12.3035),
    'dolo':                        (45.4210, 12.0941),
    'favaro':                      (45.5046, 12.2820),
    'favaro veneto':               (45.5046, 12.2820),
    "fiesse d'artico":             (45.4228, 12.0363),
    'fossalta di piave':           (45.6461, 12.5063),
    'fossalta di portogruaro':     (45.7780, 12.9139),
    "fosso'":                      (45.3787, 12.0475),
    'grisolera':                   (45.4567, 12.4592),
    'gruaro':                      (45.8198, 12.8310),
    'jesolo':                      (45.5347, 12.6538),
    'lido':                        (45.4165, 12.3706),
    'malamocco':                   (45.3720, 12.3384),
    'marcon':                      (45.5544, 12.2993),
    'marghera':                    (45.4758, 12.2248),
    'martellago':                  (45.5380, 12.1594),
    'meolo':                       (45.6086, 12.4626),
    'mestre':                      (45.4943, 12.2418),
    'mira':                        (45.4340, 12.1290),
    'mirano':                      (45.4878, 12.0859),
    'mogliano':                    (45.5786, 12.2364),
    'murano':                      (45.4585, 12.3530),
    'musile di piave':             (45.5943, 12.5231),
    'noale':                       (45.5533, 12.0724),
    'noventa di piave':            (45.6688, 12.5382),
    'stra':                        (45.4108434, 12.0377916),
    'oriago':                      (45.4506231, 12.1638481),
    'cono':                        (45.1801, 12.0787),     # OCR variant of 'cona'
    'chiirango':                   (45.4847, 12.1888),     # OCR variant of 'chirignago'
    "fossò":                       (45.3787, 12.0475),     # OCR variant of "fosso'"
    'soerè':                       (45.5801, 12.1171),     # OCR variant of 'scorzè'
    'cavenżere':                   (45.113, 12.1329),      # OCR variant of 'cavarzere'
    'torre maggiore':              (45.3914080, 11.8058487),  # Padova -- manually identified
    'toccabio':                    (45.4971455, 12.4152848),  # Torcello -- manually identified
    'pellestrina':                 (45.2723, 12.3008),
    'pianiga':                     (45.4559, 12.0301),
    'porto marghera':              (45.4724, 12.2559),
    'punta sabbioni':              (45.4477, 12.4276),
    'portogruaro':                 (45.7577, 12.7691),
    'pramaggiore':                 (45.7992, 12.7286),
    "s. dona' di piave":           (45.6264, 12.6038),
    "s. dona di piave":            (45.6264, 12.6038),  
    's. maria di sala':            (45.5010, 12.0214),
    's. michele al tagliamento':   (45.7357, 12.9940),
    's. michele del quarto':       (45.7357, 12.9940),
    's. pietro in vol':            (45.3184, 12.3167),
    's. pietro in volta':          (45.3184, 12.3167),
    's. stino di livenza':         (45.6783, 12.7722),
    'salzano':                     (45.5213, 12.1064),
    'san pietro in volta':         (45.3184, 12.3167),
    'scorzè':                      (45.5801, 12.1171),
    'spinea':                      (45.4912, 12.1650),
    'teglio veneto':               (45.8171, 12.8932),
    'tessera':                     (45.5029, 12.3276),
    'torre di mosto':              (45.6544, 12.7117),
    'treporti':                    (45.4659, 12.4568),
    'viale delle industrie':       (45.4724, 12.2559),
    'vigonovo':                    (45.3877, 12.0162),
    'zelarino':                    (45.5144, 12.2087),
    'annone venelo':               (45.7944, 12.6829),
    'campogara':                   (45.3769, 12.0921),
    'campolongn maggioro':         (45.3305, 12.0483),
    "fiesso d'artico":             (45.4228, 12.0363),
    'fossalto di portogruaro':     (45.7780, 12.9139),
    'grisolea':                    (45.4567, 12.4592),
    'mira taglio':                 (45.4340, 12.1290),
    'muslle di piave':             (45.5943, 12.5231),
    's. donà di piave':            (45.6264, 12.6038),
    'san donà di piave':           (45.6264, 12.6038),
    's. pietro di cavarzere':      (45.1130, 12.1329),
    's. stino livenza':            (45.6783, 12.7722),
    'stras':                       (45.4108434, 12.0377916),
}

# ── Nominatim accept/reject rules (used by both paths) ──────
NOMINATIM_ACCEPT_TYPES = {
    "building", "church", "place_of_worship", "cathedral", "chapel",
    "monastery", "civic", "government", "public_building",
    "hospital", "school", "university", "library", "museum",
    "theatre", "cinema", "hotel", "restaurant", "house", "apartments",
}
NOMINATIM_ACCEPT_CLASSES = {
    "building", "amenity", "tourism", "historic", "shop",
}
NOMINATIM_STREET_TYPES = {
    "residential", "service", "footway", "pedestrian", "path",
    "living_street", "unclassified", "tertiary", "secondary",
}
NOMINATIM_REJECT = {
    ("boundary", "administrative"),
    ("place", "city"),
    ("place", "town"),
    ("place", "village"),
    ("place", "municipality"),
    ("place", "neighbourhood"),
    ("place", "island"),  #for lido for ex
}

# ── Section-to-type mapping for inside-Venice prompt ────────
SECTION_HINTS = {
    "government":     "government buildings, palazzi, public institutions, tribunali, prefetture",
    "religious":      "churches, convents, monasteries, parishes — use full Italian church name",
    "professionisti": "professional offices, studios, named buildings, campi, calli",
    "industria":      "factories, warehouses, commercial premises, named buildings",
    "indice_generale":"mixed entries — use all available context from Name, Address, Location",
    "cover_ads":      "commercial advertisements, shops, hotels",
    "index":          "directory index entries — use Name as primary clue",
    "provincia":      "towns and localities in the Venice province — include town name in query",
}


def has_real_address(row):
    # True if Address has more than a town name -- street prefix or number
    addr = str(row.get("Address", "") or "").strip()
    if not addr:
        return False
    return bool(STREET_PREFIX_RE.search(addr)) or bool(CIVIC_NUM_RE.search(addr))


## Drive helpers
Shared Drive auth, folder resolution, and save/upload functions.


In [ ]:
from drive_utils import (
    get_drive_service,
    find_folder,
    get_or_create_folder,
    list_files_in_folder,
    download_text,
    upload_text,
    save_or_upload_csv,
    save_or_upload_geojson,
    list_latest_folders_local,
    list_latest_folders_drive,
    find_latest_folder_local,
    get_run_folder_id,
)


def _get_run_folder_id():
    return get_run_folder_id(DRIVE_ROOT_FOLDER, DRIVE_GEOSPATIAL_FOLDER, DRIVE_NOMINATIM_FOLDER, RUN_NAME)

def _load_output(path, svc, folder_id):
    # loads a CSV per SAVE_MODE, Drive preferred when available
    if SAVE_MODE in ("drive", "both") and svc and folder_id:
        try:
            files = svc.files().list(
                q=f"name='{path.name}' and '{folder_id}' in parents and trashed=false",
                fields="files(id,name)",
                supportsAllDrives=True, includeItemsFromAllDrives=True
            ).execute().get("files", [])
            if files:
                return pd.read_csv(io.StringIO(download_text(svc, files[0]["id"])))
        except Exception as e:
            print(f"  WARNING: Drive load failed for {path.name} — {e}")
    if SAVE_MODE in ("local", "both") and path.exists():
        try:
            return pd.read_csv(path)
        except OSError:
            print(f"  WARNING: local read failed for {path.name}")
    print(f"  WARNING: {path.name} not found anywhere")
    return pd.DataFrame()


# ── Cached folder IDs
_svc_ids       = get_drive_service()
ROOT_ID        = find_folder(_svc_ids, DRIVE_ROOT_FOLDER)
GEO_ID         = find_folder(_svc_ids, DRIVE_GEOSPATIAL_FOLDER, ROOT_ID)
OUTPUTS_ID     = find_folder(_svc_ids, DRIVE_OUTPUTS_FOLDER, GEO_ID)
NOMINATIM_ID   = find_folder(_svc_ids, DRIVE_NOMINATIM_FOLDER, GEO_ID)
FINAL_ID       = find_folder(_svc_ids, DRIVE_FINAL_FOLDER, GEO_ID)
CLEAN_PAGES_ID = find_folder(_svc_ids, DRIVE_CLEAN_PAGES_FOLDER, ROOT_ID)

print("  Drive helpers loaded.")



  Drive helpers loaded.


## Load input, classify, run tag setup
Classifies unmatched-sestiere rows into inside/outside Venice/provincia by regex.


In [ ]:
# Classification done by regex (not LLM)

FORCE_NEW_RUN = True 
# True  → always create a new timestamped folder (use when starting a real new run)
# False → reuse the most recent existing folder (use when restoring kernel for debug)

def find_latest_unmatched_local():
    # most recent local run with an unmatched CSV
    try:
        for folder in list_latest_folders_local(Path(GEOCODER_OUTPUT_DIR), "all_pages_"):
            candidates = list(folder.glob("*_unmatched.csv"))
            if candidates:
                return candidates[0], folder.name
    except Exception as e:
        print(f"  WARNING: local scan failed — {e}")
    return None, None


def find_latest_unmatched_drive():
    # most recent Drive run with an unmatched CSV
    try:
        service = get_drive_service()
        if service is None:
            return None, None
        root_id = find_folder(service, DRIVE_ROOT_FOLDER)
        geo_id  = find_folder(service, DRIVE_GEOSPATIAL_FOLDER, root_id)
        out_id  = find_folder(service, DRIVE_OUTPUTS_FOLDER, geo_id)
        for folder in list_latest_folders_drive(service, out_id, "all_pages_"):
            run_files = list_files_in_folder(service, folder["id"])
            unmatched = next(
                (f for f in run_files if "_unmatched" in f["name"]),
                None
            )
            if unmatched:
                return unmatched["id"], folder["name"]
    except Exception as e:
        print(f"  WARNING: Drive scan failed — {e}")
    return None, None


# ── Load unmatched CSV ───────────────────────────────────────
print("=== LOADING UNMATCHED ROWS FROM SESTIERE GEOCODER ===")
df_unmatched    = None
source_run_name = None

unmatched_path, source_run_name = find_latest_unmatched_local()
if unmatched_path is not None:
    try:
        df_unmatched = pd.read_csv(unmatched_path, dtype=str)
        print(f"  Local: {unmatched_path}")
    except Exception as e:
        print(f"  WARNING: could not read local file — {e}")
        df_unmatched = None

if df_unmatched is None:
    print("  Trying Drive ...")
    file_id, source_run_name = find_latest_unmatched_drive()
    if file_id is not None:
        try:
            text = download_text(get_drive_service(), file_id)
            df_unmatched = pd.read_csv(io.StringIO(text), dtype=str)
            print(f"  Drive: {source_run_name}")
        except Exception as e:
            print(f"  WARNING: could not read Drive file — {e}")

if df_unmatched is None:
    print("\n  ERROR: no unmatched CSV found locally or on Drive.")
    print("  Run notebook 05 (geocoder) first.")
    raise SystemExit(0)

print(f"  Total unmatched rows loaded: {len(df_unmatched)}")

# ── Step 1 — Regex classification ───────────────────────────
# OUTSIDE_VENICE_RE fires on Address OR Location → outside_venice
# No match → inside_venice

def classify_row(row):
    for field in ["Address", "Location"]:
        val = str(row.get(field, "") or "").strip()
        if val and OUTSIDE_VENICE_RE.search(val):
            return "outside_venice"
    return "inside_venice"

df_unmatched["nom_classification"] = df_unmatched.apply(classify_row, axis=1)

df_outside = df_unmatched[
    df_unmatched["nom_classification"] == "outside_venice"
].copy().reset_index(drop=True)

df_inside = df_unmatched[
    df_unmatched["nom_classification"] == "inside_venice"
].copy().reset_index(drop=True)

# Provincia section — flag separately for stats
PROVINCIA_PAGES = set(range(494, 572))

print(f"\n  Classification results:")
print(f"    outside_venice : {len(df_outside)} rows")
print(f"    inside_venice  : {len(df_inside)} rows")

# ── Loas provincia per-page CSVs from clean_pages ────────────
print("\n  Loading provincia per-page CSVs from clean_pages ...")

PROVINCIA_PAGES_SET = set(range(494, 571))

def load_provincia_pages_local():
    base = Path(CLEAN_PAGES_DIR)
    if not base.exists():
        return None
    rows = []
    for page_num in sorted(PROVINCIA_PAGES_SET):
        page_dir = base / f"page_{page_num}"
        csv_candidates = list(page_dir.glob("*_semantic.csv")) if page_dir.exists() else []
        if not csv_candidates:
            continue
        try:
            df_page = pd.read_csv(csv_candidates[0], dtype=str)
            df_page["page_num"] = str(page_num)
            rows.append(df_page)
        except Exception as e:
            print(f"  WARNING: could not read page_{page_num} — {e}")
    return pd.concat(rows, ignore_index=True) if rows else None

def load_provincia_pages_drive():
    try:
        svc     = get_drive_service()
        root_id = find_folder(svc, DRIVE_ROOT_FOLDER)
        cp_id   = find_folder(svc, DRIVE_CLEAN_PAGES_FOLDER, root_id)
        if not cp_id:
            return None
        rows = []
        for page_num in sorted(PROVINCIA_PAGES_SET):
            page_folder_id = find_folder(svc, f"page_{page_num}", cp_id)
            if not page_folder_id:
                continue
            files = list_files_in_folder(svc, page_folder_id)
            sem = next((f for f in files if f["name"].endswith("_semantic.csv")), None)
            if not sem:
                continue
            try:
                text = download_text(svc, sem["id"])
                df_page = pd.read_csv(io.StringIO(text), dtype=str)
                df_page["page_num"] = str(page_num)
                rows.append(df_page)
            except Exception as e:
                print(f"  WARNING: could not read page_{page_num} from Drive — {e}")
        return pd.concat(rows, ignore_index=True) if rows else None
    except Exception as e:
        print(f"  WARNING: Drive load failed — {e}")
        return None

df_provincia_raw = load_provincia_pages_local()
if df_provincia_raw is None:
    print("  Local not found — trying Drive ...")
    df_provincia_raw = load_provincia_pages_drive()

if df_provincia_raw is not None:
    print(f"  Loaded {len(df_provincia_raw)} provincia rows "
          f"from {df_provincia_raw['page_num'].nunique()} pages")

    df_provincia = pd.DataFrame()
    df_provincia["Name"]             = df_provincia_raw.get("Name",             pd.Series(dtype=str)).fillna("")
    df_provincia["Address"]          = df_provincia_raw.get("Address",          pd.Series(dtype=str)).fillna("")
    df_provincia["Location"]         = df_provincia_raw.get("Provincia",        pd.Series(dtype=str)).fillna("").str.title()
    df_provincia["section"]          = df_provincia_raw.get("Section",          pd.Series(dtype=str)).fillna("provincia")
    df_provincia["Role"]             = df_provincia_raw.get("Role_or_Profession", pd.Series(dtype=str)).fillna("")
    df_provincia["page_num"]         = df_provincia_raw["page_num"]
    df_provincia["nom_classification"] = "outside_venice"
    df_provincia["is_provincia"]       = True
    df_provincia["provincia_town"]     = df_provincia_raw.get(
        "Provincia", pd.Series(dtype=str)
    ).fillna("").str.lower().str.strip()

    print(f"  df_provincia ready: {len(df_provincia)} rows")
    print(f"  Towns: {df_provincia['provincia_town'].nunique()} unique")
    
    # Cache locally for faster future loads
    try:
        _prov_cache = Path(LOCAL_ALMANAC_ROOT) / ".cache" / "df_provincia.csv"
        _prov_cache.parent.mkdir(exist_ok=True)
        df_provincia.to_csv(_prov_cache, index=False)
    except OSError:
        pass  
else:
    print("  WARNING: no provincia pages found")
    df_provincia = pd.DataFrame()

# ── Run tag + output paths ───────────────────────────────────────────
_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")

if FORCE_NEW_RUN:
    RUN_NAME = f"nominatim_{_TIMESTAMP}"
    RUN_DIR  = Path(NOMINATIM_OUTPUT_DIR) / RUN_NAME
    print(f"  FORCE_NEW_RUN=True — creating new folder: {RUN_NAME}")
else:
    _existing_dir = find_latest_folder_local(NOMINATIM_OUTPUT_DIR, "nominatim_")
    _existing_name = _existing_dir.name if _existing_dir else None
    if _existing_name:
        RUN_NAME = _existing_name
        RUN_DIR  = _existing_dir
        print(f"  FORCE_NEW_RUN=False — reusing existing folder: {RUN_NAME}")
    else:
        RUN_NAME = f"nominatim_{_TIMESTAMP}"
        RUN_DIR  = Path(NOMINATIM_OUTPUT_DIR) / RUN_NAME
        print(f"  No existing folder found — creating: {RUN_NAME}")

RUN_DIR.mkdir(parents=True, exist_ok=True)

# ── Both inside and outside share the same parent folder ─────────────
OUTSIDE_RUN_NAME = f"nominatim_{_TIMESTAMP}"   # filename tag, generated fresh each time
INSIDE_RUN_NAME  = f"nominatim_{_TIMESTAMP}"   
OUTSIDE_RUN_DIR  = RUN_DIR
INSIDE_RUN_DIR   = RUN_DIR
LOCAL_RUN_DIR    = RUN_DIR                
OUT_PROVINCIA_NOM_CSV      = RUN_DIR / f"nominatim_provincia_matched_{OUTSIDE_RUN_NAME}.csv"
OUT_PROVINCIA_NOM_GEOJSON  = RUN_DIR / f"nominatim_provincia_matched_{OUTSIDE_RUN_NAME}.geojson"
OUT_PROVINCIA_CENTROID_CSV = RUN_DIR / f"centroid_provincia_matched_{OUTSIDE_RUN_NAME}.csv"
OUT_PROVINCIA_CENTROID_GEOJSON = RUN_DIR / f"centroid_provincia_matched_{OUTSIDE_RUN_NAME}.geojson"
OUT_PROVINCIA_UNMATCHED    = RUN_DIR / f"nominatim_provincia_unmatched_{OUTSIDE_RUN_NAME}.csv"
OUT_DECISIONS_PROVINCIA    = RUN_DIR / f"{OUTSIDE_RUN_NAME}_nominatim_decisions_provincia.csv"

print(f"  Run folder : {RUN_NAME}")
print(f"  Run dir    : {RUN_DIR}")
print(f"  Source geocoder run : {source_run_name}")

=== LOADING UNMATCHED ROWS FROM SESTIERE GEOCODER ===
  Local: geospatial/outputs/all_pages_20260721_1901/all_pages_20260721_1901_unmatched_sestiere.csv
  Total unmatched rows loaded: 10228

  Classification results:
    outside_venice : 5515 rows
    inside_venice  : 4713 rows

  Loading provincia per-page CSVs from clean_pages ...
  Loaded 9132 provincia rows from 76 pages
  df_provincia ready: 9132 rows
  Towns: 57 unique
  FORCE_NEW_RUN=True — creating new folder: nominatim_20260807_1750
  Run folder : nominatim_20260807_1750
  Run dir    : geospatial/nominatim_outputs/nominatim_20260807_1750
  Source geocoder run : all_pages_20260721_1901


## Outside-Venice geocoding


In [ ]:
#  RUN_OUTSIDE = True/False to enable/disable
#
#  Tiers:
#    5   — Nominatim matched real address
#    5f  — Nominatim matched, flagged/ambiguous
#    7a  — Town centroid, no real address
#    7b  — Town centroid, had real address but Nominatim failed
#
#  Outputs:
#    nominatim_geocoded_matched.csv/.geojson (tiers 5, 5f)
#    outside_venice_centroid_matched.csv/.geojson (tiers 7a, 7b)
#    nominatim_unmatched.csv (failed entirely)
#    nominatim_decisions_outside.csv (audit log)
#
#  Skip condition: if decisions_outside.csv exists → reload

if RUN_OUTSIDE:
    print("\n=== CELL 4 — OUTSIDE VENICE GEOCODING ===")
    print(f"  Rows to process: {len(df_outside)}")

    REUSE_OUTSIDE_DECISIONS = False
    # True  → load existing decisions file, zero API cost
    # False → re-run LLM + Nominatim from scratch

    REAPPLY_AND_SAVE = False
    # REUSE_OUTSIDE_DECISIONS=True  + REAPPLY_AND_SAVE=False → fast load outputs into memory only
    # REUSE_OUTSIDE_DECISIONS=True  + REAPPLY_AND_SAVE=True  → re-apply decisions, re-save files (in case of other changes not related to LLM decisions)
    # REUSE_OUTSIDE_DECISIONS=False + (any)                  → full fresh API run

    # ── Generate or reuse run name and output paths ──────────
    if REUSE_OUTSIDE_DECISIONS:
        # Scan run folder for most recent decisions file
        _existing_dec = sorted(
            RUN_DIR.glob("*_nominatim_decisions_outside.csv"),
            key=lambda f: f.stat().st_mtime,
            reverse=True
        ) if RUN_DIR.exists() else []
        
        if _existing_dec:
            # Extract run name from existing decisions filename
            OUTSIDE_RUN_NAME = _existing_dec[0].name.replace(
                "_nominatim_decisions_outside.csv", ""
            )
            print(f"  REUSE_OUTSIDE_DECISIONS=True — reusing: {OUTSIDE_RUN_NAME}")
        else:
            # No existing file on local — try Drive
            _svc_scan, _scan_folder_id = _get_run_folder_id()
            _drive_decs = []
            if _svc_scan and _scan_folder_id:
                _drive_decs = [
                    f for f in list_files_in_folder(_svc_scan, _scan_folder_id)
                    if "_nominatim_decisions_outside.csv" in f["name"]
                ]
            if _drive_decs:
                _latest_dec = max(_drive_decs, key=lambda f: f["createdTime"])
                OUTSIDE_RUN_NAME = _latest_dec["name"].replace(
                    "_nominatim_decisions_outside.csv", ""
                )
                print(f"  REUSE_OUTSIDE_DECISIONS=True — reusing from Drive: {OUTSIDE_RUN_NAME}")
            else:
                # No decisions file anywhere — fall back to fresh run
                print(f"  REUSE_OUTSIDE_DECISIONS=True but no decisions file found — running fresh")
                OUTSIDE_RUN_NAME = f"nominatim_{datetime.now().strftime('%Y%m%d_%H%M')}"
    else:
        OUTSIDE_RUN_NAME = f"nominatim_{datetime.now().strftime('%Y%m%d_%H%M')}"
        print(f"  REUSE_OUTSIDE_DECISIONS=False — new run: {OUTSIDE_RUN_NAME}")

    # ── Output paths (always set after run name resolved) ────
    OUT_OUTSIDE_NOM_CSV     = RUN_DIR / f"nominatim_outside_matched_{OUTSIDE_RUN_NAME}.csv"
    OUT_OUTSIDE_NOM_GEOJSON = RUN_DIR / f"nominatim_outside_matched_{OUTSIDE_RUN_NAME}.geojson"
    OUT_CENTROID_CSV        = RUN_DIR / f"outside_venice_centroid_matched_{OUTSIDE_RUN_NAME}.csv"
    OUT_CENTROID_GEOJSON    = RUN_DIR / f"outside_venice_centroid_matched_{OUTSIDE_RUN_NAME}.geojson"
    OUT_OUTSIDE_UNMATCHED   = RUN_DIR / f"nominatim_outside_unmatched_{OUTSIDE_RUN_NAME}.csv"
    OUT_DECISIONS_OUTSIDE   = RUN_DIR / f"{OUTSIDE_RUN_NAME}_nominatim_decisions_outside.csv"


    # ── Fast load (kernel restore only) ─────────────────────
    if REUSE_OUTSIDE_DECISIONS and not REAPPLY_AND_SAVE:
        print(f"\n  REAPPLY_AND_SAVE=False — loading output files directly ...")
        _svc_load, _load_folder_id = _get_run_folder_id()
        nom_outside  = _load_output(OUT_OUTSIDE_NOM_CSV,  _svc_load, _load_folder_id)
        cent_outside = _load_output(OUT_CENTROID_CSV,      _svc_load, _load_folder_id)
        fail_outside = _load_output(OUT_OUTSIDE_UNMATCHED, _svc_load, _load_folder_id)
        print(f"  Nominatim outside : {len(nom_outside)} rows")
        print(f"  Centroids         : {len(cent_outside)} rows")
        print(f"  Failed outside    : {len(fail_outside)} rows")
        print(f"\n=== CELL 4 DONE (fast load) ===")
    else:
        # ── Helper functions ─────────────────────────────────

        def find_outside_trigger(row):
            # (trigger_word, field) for the first OUTSIDE_VENICE_RE match
            for field in ["Address", "Location"]:
                val = str(row.get(field, "") or "").strip()
                if not val:
                    continue
                m = OUTSIDE_VENICE_RE.search(val)
                if m:
                    return m.group(0).lower(), field
            return None, None

        def nominatim_geocode_full(query):
            # calls Nominatim, returns the full result dict or None
            params = urllib.parse.urlencode({
                "q":              query,
                "format":         "json",
                "limit":          1,
                "countrycodes":   "it",
                "addressdetails": 0,
            })
            url = f"https://nominatim.openstreetmap.org/search?{params}"
            headers = {"User-Agent": "VeniceAlmanac1947/1.0 (thesis research)"}
            try:
                req = urllib.request.Request(url, headers=headers)
                with urllib.request.urlopen(req, timeout=10) as resp:
                    data = json.loads(resp.read().decode())
                if data:
                    hit = data[0]
                    return {
                        "lat":          float(hit["lat"]),
                        "lon":          float(hit["lon"]),
                        "osm_type":     hit.get("type", ""),
                        "osm_class":    hit.get("class", ""),
                        "display_name": hit.get("display_name", ""),
                    }
            except Exception:
                pass
            return None

        def nominatim_result_is_acceptable(result):
            # (accepted, flagged)
            if result is None:
                return False, False
            osm_type  = result.get("osm_type", "")
            osm_class = result.get("osm_class", "")
            if (osm_class, osm_type) in NOMINATIM_REJECT:
                return False, False
            if osm_type in NOMINATIM_ACCEPT_TYPES:
                return True, False
            if osm_class in NOMINATIM_ACCEPT_CLASSES:
                return True, False
            if osm_type in NOMINATIM_STREET_TYPES or osm_class == "highway":
                return True, True
            return True, True

        def llm_clean_address(row, trigger_word, client):
            # asks GPT-4o to extract a clean geocodable address string
            addr = str(row.get("Address", "") or "")
            loc  = str(row.get("Location", "") or "")
            name = str(row.get("Name", "") or "")
            prompt = f"""You are cleaning addresses from a 1947 Italian commercial almanac.
    This row refers to a location in or near "{trigger_word}" (outside Venice proper).

    Row data:
    Name: {name}
    Address: {addr}
    Location: {loc}

    Extract the single best geocodable address string.
    Return ONLY the address as a plain string, e.g. "Via Cappuccina 129, Mestre".
    If the address is just a town name with no further detail, return just the town name.
    IMPORTANT: if the address contains a street name and number (e.g. 'Via Manin 39'), you MUST include both the street name and number in your response.
    Never return just a town name if a street address is present. Only return just the town name if there is genuinely no street information at all.
    If you cannot determine an address, return null.
    No explanation, no JSON, just the address string or null."""
            try:
                response = client.chat.completions.create(
                    model="gpt-4o",
                    max_tokens=100,
                    temperature=0,
                    messages=[{"role": "user", "content": prompt}]
                )
                result = response.choices[0].message.content.strip()
                return None if result.lower() in ("null", "none", "") else result
            except Exception as e:
                print(f"    LLM error: {e}")
                return None

        # ── Skip condition ───────────────────────────────────────
        if OUT_DECISIONS_OUTSIDE.exists() or REUSE_OUTSIDE_DECISIONS:
            # Try to load existing decisions file
            _loaded = False
            if OUT_DECISIONS_OUTSIDE.exists():
                try:
                    outside_decisions = pd.read_csv(OUT_DECISIONS_OUTSIDE)
                    print(f"\n  Decisions file found — loading {OUT_DECISIONS_OUTSIDE.name}")
                    print(f"  (no API calls) — {len(outside_decisions)} rows")
                    _loaded = True
                except OSError:
                    print(f"  Local read failed (Drive-mounted) — downloading via API ...")

            if not _loaded:
                # Try Drive fallback
                _svc_tmp, _run_folder_id_tmp = _get_run_folder_id()
                _dec_files = _svc_tmp.files().list(
                    q=f"'{_run_folder_id_tmp}' in parents and trashed=false"
                    f" and name='{OUT_DECISIONS_OUTSIDE.name}'",
                    fields="files(id,name)",
                    supportsAllDrives=True, includeItemsFromAllDrives=True
                ).execute().get("files", [])
                if _dec_files:
                    outside_decisions = pd.read_csv(
                        io.StringIO(download_text(_svc_tmp, _dec_files[0]["id"]))
                    )
                    print(f"  Loaded from Drive: {len(outside_decisions)} rows")
                    _loaded = True

            if not _loaded:
                print(f"\n  No decisions file found — running LLM + Nominatim ...")
                # fall through to API block below
                _run_api = True
            else:
                _run_api = False

        else:
            print(f"\n  No decisions file — running LLM + Nominatim ...")
            _run_api = True

        if _run_api:
            with open(Path(OPENAI_KEY_FILE)) as f:
                _key_params = dict(
                    v.strip().split("=", 1) for v in f if "=" in v
                )
            llm_client      = openai.OpenAI(api_key=_key_params["api_key"])
            nominatim_cache = {}
            outside_records = []
            llm_calls = nom_calls = 0

            for idx, row in df_outside.iterrows():
                trigger_word, trigger_field = find_outside_trigger(row)

                if trigger_word is None:
                    outside_records.append({
                        "source_index":        idx,
                        "page_num":            row.get("page_num"),
                        "Name":                str(row.get("Name", "") or ""),
                        "Address":             str(row.get("Address", "") or ""),
                        "Location":            str(row.get("Location", "") or ""),
                        "section":             str(row.get("section", "") or ""),
                        "trigger_word":        None,
                        "has_real_address":    False,
                        "llm_cleaned":         None,
                        "nominatim_query":     None,
                        "nom_lat":             None,
                        "nom_lon":             None,
                        "nom_type":            None,
                        "nom_class":           None,
                        "nom_display":         None,
                        "accepted":            False,
                        "flagged":             False,
                        "geocoding_tier":      None,
                        "match_confidence":    "outside_venice_no_trigger",
                        "match_note":          "no OUTSIDE_VENICE_RE match found",
                    })
                    continue

                centroid        = TOWN_CENTROIDS.get(trigger_word)
                real_addr       = has_real_address(row)
                cleaned         = None
                query           = None
                nom_result      = None
                accepted        = False
                flagged         = False
                tier            = None
                confidence      = None
                note            = None

                if real_addr:
                    cleaned   = llm_clean_address(row, trigger_word, llm_client)
                    llm_calls += 1
                    time.sleep(0.3)

                    query = (
                        f"{cleaned}, Italy"
                        if cleaned and "italy" not in cleaned.lower()
                        else (cleaned or f"{trigger_word}, Italy")
                    )

                    if query not in nominatim_cache:
                        nom_result = nominatim_geocode_full(query)
                        nominatim_cache[query] = nom_result
                        nom_calls += 1
                        time.sleep(1.1)
                    else:
                        nom_result = nominatim_cache[query]

                    accepted, flagged = nominatim_result_is_acceptable(nom_result)

                    if accepted and nom_result:
                        tier       = "5f" if flagged else "5"
                        confidence = "nominatim_outside_flagged" if flagged else "nominatim_outside"
                        note       = (
                            f"Nominatim outside: {query} | "
                            f"type={nom_result['osm_type']} class={nom_result['osm_class']}"
                            + (" | FLAGGED" if flagged else "")
                        )
                    elif centroid:
                        # Had real address but Nominatim failed → tier 7b
                        tier       = "7b"
                        confidence = "centroid_nominatim_failed"
                        note       = f"Nominatim failed for '{query}' — centroid for {trigger_word}"
                        nom_result = None
                        accepted   = False
                    else:
                        tier       = None
                        confidence = "outside_venice_failed"
                        note       = f"Nominatim failed and no centroid for {trigger_word}"
                        nom_result = None
                        accepted   = False
                else:
                    # No real address → centroid directly → tier 7a
                    if centroid:
                        tier       = "7a"
                        confidence = "centroid_direct"
                        note       = f"Town name only — centroid for {trigger_word}"
                    else:
                        tier       = None
                        confidence = "outside_venice_failed"
                        note       = f"No centroid for {trigger_word}"

                outside_records.append({
                    "source_index":     idx,
                    "page_num":         row.get("page_num"),
                    "Name":             str(row.get("Name", "") or ""),
                    "Address":          str(row.get("Address", "") or ""),
                    "Location":         str(row.get("Location", "") or ""),
                    "section":          str(row.get("section", "") or ""),
                    "trigger_word":     trigger_word,
                    "has_real_address": real_addr,
                    "llm_cleaned":      cleaned,
                    "nominatim_query":  query,
                    "nom_lat":          nom_result["lat"]          if nom_result and accepted else None,
                    "nom_lon":          nom_result["lon"]          if nom_result and accepted else None,
                    "nom_type":         nom_result["osm_type"]     if nom_result else None,
                    "nom_class":        nom_result["osm_class"]    if nom_result else None,
                    "nom_display":      nom_result["display_name"] if nom_result else None,
                    "accepted":         accepted,
                    "flagged":          flagged,
                    "geocoding_tier":   tier,
                    "match_confidence": confidence,
                    "match_note":       note,
                })

                if (idx + 1) % 100 == 0:
                    print(f"  Progress: {idx+1}/{len(df_outside)} | "
                        f"LLM={llm_calls} Nom={nom_calls}")

            outside_decisions = pd.DataFrame(outside_records)
            LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
            outside_decisions.to_csv(OUT_DECISIONS_OUTSIDE, index=False)
            print(f"\n  Decisions saved → {OUT_DECISIONS_OUTSIDE.name} "
                f"({len(outside_decisions)} rows)")
            print(f"  LLM calls: {llm_calls} | Nominatim calls: {nom_calls}")

        # ── Apply decisions ──────────────────────────────────────
        print("\n  Applying outside-Venice decisions ...")

        # Merge decisions back onto df_outside
        df_outside_result = df_outside.copy()
        df_outside_result["geocoding_tier"]   = outside_decisions["geocoding_tier"].values
        df_outside_result["match_confidence"] = outside_decisions["match_confidence"].values
        df_outside_result["match_note"]       = outside_decisions["match_note"].values
        df_outside_result["nominatim_query"]  = outside_decisions["nominatim_query"].values
        df_outside_result["llm_cleaned_address"] = outside_decisions["llm_cleaned"].values
        df_outside_result["nominatim_flagged"]   = outside_decisions["flagged"].values

        # Assign coordinates — Nominatim rows get nom_lat/lon,
        # centroid rows get TOWN_CENTROIDS values
        def get_lat(row_dec):
            if pd.notna(row_dec.get("nom_lat")) and row_dec.get("accepted"):
                return float(row_dec["nom_lat"])
            tw = row_dec.get("trigger_word")
            tier = str(row_dec.get("geocoding_tier", ""))
            if tier in ("7a", "7b") and tw and tw in TOWN_CENTROIDS:
                return TOWN_CENTROIDS[tw][0]
            return None

        def get_lon(row_dec):
            if pd.notna(row_dec.get("nom_lon")) and row_dec.get("accepted"):
                return float(row_dec["nom_lon"])
            tw = row_dec.get("trigger_word")
            tier = str(row_dec.get("geocoding_tier", ""))
            if tier in ("7a", "7b") and tw and tw in TOWN_CENTROIDS:
                return TOWN_CENTROIDS[tw][1]
            return None

        df_outside_result["matched_lat"] = outside_decisions.apply(get_lat, axis=1)
        df_outside_result["matched_lon"] = outside_decisions.apply(get_lon, axis=1)

        # Split into Nominatim matched vs centroid vs unmatched
        nom_outside = df_outside_result[df_outside_result["geocoding_tier"].isin(["5", "5f"])].copy()
        cent_outside = df_outside_result[df_outside_result["geocoding_tier"].isin(["7a", "7b"])].copy()
        fail_outside = df_outside_result[df_outside_result["matched_lat"].isna()].copy()

        print(f"  Tier 5  (Nominatim address)        : {(outside_decisions['geocoding_tier'] == '5').sum()}")
        print(f"  Tier 5f (Nominatim, flagged)        : {(outside_decisions['geocoding_tier'] == '5f').sum()}")
        print(f"  Tier 7a (centroid direct)           : {(outside_decisions['geocoding_tier'] == '7a').sum()}")
        print(f"  Tier 7b (centroid, Nominatim failed): {(outside_decisions['geocoding_tier'] == '7b').sum()}")
        print(f"  Failed  (no coordinates)            : {len(fail_outside)}")

        # ── Save outside-Venice outputs ──────────────────────────
        _svc, _run_folder_id = _get_run_folder_id()

        # Nominatim matched (tiers 5, 5f) → nominatim_outside_matched
        if not nom_outside.empty:
            save_or_upload_csv(nom_outside, OUT_OUTSIDE_NOM_CSV, _svc, _run_folder_id)
            gdf_nom_out = gpd.GeoDataFrame(
                nom_outside,
                geometry=gpd.points_from_xy(
                    nom_outside["matched_lon"].astype(float),
                    nom_outside["matched_lat"].astype(float),
                ),
                crs="EPSG:4326",
            )
            save_or_upload_geojson(gdf_nom_out, OUT_OUTSIDE_NOM_GEOJSON, _svc, _run_folder_id)
            print(f"\n  Nominatim geocoded (outside) → {OUT_OUTSIDE_NOM_CSV.name} ({len(nom_outside)} rows)")

        # Centroid rows (tiers 7a, 7b) → outside_venice_centroid_matched
        if not cent_outside.empty:
            save_or_upload_csv(cent_outside, OUT_CENTROID_CSV, _svc, _run_folder_id)
            gdf_cent = gpd.GeoDataFrame(
                cent_outside,
                geometry=gpd.points_from_xy(
                    cent_outside["matched_lon"].astype(float),
                    cent_outside["matched_lat"].astype(float),
                ),
                crs="EPSG:4326",
            )
            save_or_upload_geojson(gdf_cent, OUT_CENTROID_GEOJSON, _svc, _run_folder_id)
            print(f"  Centroid placed  → {OUT_CENTROID_CSV.name} ({len(cent_outside)} rows)")

        # Outside failures — separate file, NOT shared with Cell 5
        save_or_upload_csv(fail_outside, OUT_OUTSIDE_UNMATCHED, _svc, _run_folder_id)
        print(f"  Outside unmatched → {OUT_OUTSIDE_UNMATCHED.name} ({len(fail_outside)} rows)")

        # Decisions log
        save_or_upload_csv(outside_decisions, OUT_DECISIONS_OUTSIDE, _svc, _run_folder_id)
        print(f"  Decisions log    → {OUT_DECISIONS_OUTSIDE.name}")

        print(f"\n=== CELL 4 DONE ===")
        # Aliases for consistent naming across notebook
        matched_outside  = nom_outside
        centroid_outside = cent_outside
        unmatched_outside = fail_outside

else:
    print("\n  Cell 4 skipped (RUN_OUTSIDE=False)")
    nom_outside  = pd.DataFrame()
    cent_outside = pd.DataFrame()
    fail_outside = pd.DataFrame()
    #aliases
    matched_outside   = nom_outside
    centroid_outside  = cent_outside
    unmatched_outside = fail_outside




=== CELL 4 — OUTSIDE VENICE GEOCODING ===
  Rows to process: 5515
  REUSE_OUTSIDE_DECISIONS=False — new run: nominatim_20260721_1924

  No decisions file — running LLM + Nominatim ...
  Progress: 100/5515 | LLM=28 Nom=22
  Progress: 200/5515 | LLM=98 Nom=80
  Progress: 300/5515 | LLM=196 Nom=169
  Progress: 400/5515 | LLM=291 Nom=250
  Progress: 500/5515 | LLM=380 Nom=332
  Progress: 600/5515 | LLM=471 Nom=407
  Progress: 700/5515 | LLM=567 Nom=482
  Progress: 800/5515 | LLM=664 Nom=557
  Progress: 900/5515 | LLM=749 Nom=620
  Progress: 1000/5515 | LLM=840 Nom=689
  Progress: 1100/5515 | LLM=936 Nom=774
  Progress: 1200/5515 | LLM=1031 Nom=860
  Progress: 1300/5515 | LLM=1117 Nom=932
  Progress: 1400/5515 | LLM=1210 Nom=1009
  Progress: 1500/5515 | LLM=1302 Nom=1073
  Progress: 1600/5515 | LLM=1396 Nom=1151
  Progress: 1700/5515 | LLM=1493 Nom=1226
  Progress: 1800/5515 | LLM=1590 Nom=1307
  Progress: 1900/5515 | LLM=1678 Nom=1375
  Progress: 2000/5515 | LLM=1770 Nom=1445
  Progress: 

## Provincia geocoding


In [11]:
#  CELL 4B — PROVINCIA GEOCODING
#  RUN_PROVINCIA = True/False to enable/disable
#
#  Tiers:
#    5   — Nominatim matched real address
#    5f  — Nominatim matched, flagged/ambiguous
#    7a  — Town centroid, no real address
#    7b  — Town centroid, had real address but Nominatim failed
#
#  Outputs (separate from Cell 4):
#    nominatim_provincia_matched.csv/.geojson
#    centroid_provincia_matched.csv/.geojson
#    nominatim_provincia_unmatched.csv
#    nominatim_decisions_provincia.csv

RUN_PROVINCIA        = True
REUSE_PROVINCIA_DECISIONS = False
REAPPLY_PROVINCIA_AND_SAVE = True  

if RUN_PROVINCIA and not df_provincia.empty:
    print("\n=== CELL 4B — PROVINCIA GEOCODING ===")
    print(f"  Rows to process: {len(df_provincia)}")

    # ── Generate or reuse run name ────────────────────────────
    if REUSE_PROVINCIA_DECISIONS:
        _existing_dec = sorted(
            RUN_DIR.glob("*_nominatim_decisions_provincia.csv"),
            key=lambda f: f.stat().st_mtime, reverse=True
        ) if RUN_DIR.exists() else []

        if _existing_dec:
            _PROV_RUN_NAME = _existing_dec[0].name.replace(
                "_nominatim_decisions_provincia.csv", ""
            )
            print(f"  REUSE=True — reusing local: {_PROV_RUN_NAME}")
        else:
            _svc_scan, _scan_folder_id = _get_run_folder_id()
            _drive_decs = [
                f for f in list_files_in_folder(_svc_scan, _scan_folder_id)
                if "_nominatim_decisions_provincia.csv" in f["name"]
            ] if _svc_scan and _scan_folder_id else []

            if _drive_decs:
                _latest = max(_drive_decs, key=lambda f: f["createdTime"])
                _PROV_RUN_NAME = _latest["name"].replace(
                    "_nominatim_decisions_provincia.csv", ""
                )
                print(f"  REUSE=True — reusing from Drive: {_PROV_RUN_NAME}")
            else:
                _PROV_RUN_NAME = f"nominatim_{datetime.now().strftime('%Y%m%d_%H%M')}"
                print(f"  No decisions file found — fresh run: {_PROV_RUN_NAME}")
    else:
        _PROV_RUN_NAME = f"nominatim_{datetime.now().strftime('%Y%m%d_%H%M')}"
        print(f"  REUSE=False — fresh run: {_PROV_RUN_NAME}")

    # ── Output paths ──────────────────────────────────────────
    OUT_PROVINCIA_NOM_CSV          = RUN_DIR / f"nominatim_provincia_matched_{_PROV_RUN_NAME}.csv"
    OUT_PROVINCIA_NOM_GEOJSON      = RUN_DIR / f"nominatim_provincia_matched_{_PROV_RUN_NAME}.geojson"
    OUT_PROVINCIA_CENTROID_CSV     = RUN_DIR / f"centroid_provincia_matched_{_PROV_RUN_NAME}.csv"
    OUT_PROVINCIA_CENTROID_GEOJSON = RUN_DIR / f"centroid_provincia_matched_{_PROV_RUN_NAME}.geojson"
    OUT_PROVINCIA_UNMATCHED        = RUN_DIR / f"nominatim_provincia_unmatched_{_PROV_RUN_NAME}.csv"
    OUT_DECISIONS_PROVINCIA        = RUN_DIR / f"{_PROV_RUN_NAME}_nominatim_decisions_provincia.csv"

    # ── Fast load ─────────────────────────────────────────────
    if REUSE_PROVINCIA_DECISIONS and not REAPPLY_PROVINCIA_AND_SAVE:
        print(f"\n  REAPPLY=False — loading output files directly ...")
        _svc_load, _load_folder_id = _get_run_folder_id()
        nom_provincia      = _load_output(OUT_PROVINCIA_NOM_CSV,      _svc_load, _load_folder_id)
        cent_provincia     = _load_output(OUT_PROVINCIA_CENTROID_CSV,  _svc_load, _load_folder_id)
        fail_provincia     = _load_output(OUT_PROVINCIA_UNMATCHED,     _svc_load, _load_folder_id)
        print(f"  Nominatim provincia : {len(nom_provincia)} rows")
        print(f"  Centroid provincia  : {len(cent_provincia)} rows")
        print(f"  Failed provincia    : {len(fail_provincia)} rows")
        print(f"\n=== CELL 4B DONE (fast load) ===")

    else:
        # ── Skip condition ────────────────────────────────────
        if OUT_DECISIONS_PROVINCIA.exists() or REUSE_PROVINCIA_DECISIONS:
            _loaded = False
            if OUT_DECISIONS_PROVINCIA.exists():
                try:
                    provincia_decisions = pd.read_csv(OUT_DECISIONS_PROVINCIA)
                    print(f"  Decisions loaded locally: {len(provincia_decisions)} rows")
                    _loaded = True
                except OSError:
                    print(f"  Local read failed — trying Drive ...")

            if not _loaded:
                _svc_tmp, _folder_tmp = _get_run_folder_id()
                _dec_files = _svc_tmp.files().list(
                    q=f"'{_folder_tmp}' in parents and trashed=false"
                      f" and name='{OUT_DECISIONS_PROVINCIA.name}'",
                    fields="files(id,name)",
                    supportsAllDrives=True, includeItemsFromAllDrives=True
                ).execute().get("files", [])
                if _dec_files:
                    provincia_decisions = pd.read_csv(
                        io.StringIO(download_text(_svc_tmp, _dec_files[0]["id"]))
                    )
                    print(f"  Loaded from Drive: {len(provincia_decisions)} rows")
                    _loaded = True

            _run_api = not _loaded
        else:
            _run_api = True

        if _run_api:
            print(f"\n  Running LLM + Nominatim for provincia rows ...")
            with open(Path(OPENAI_KEY_FILE)) as f:
                _key_params = dict(v.strip().split("=", 1) for v in f if "=" in v)
            llm_client      = openai.OpenAI(api_key=_key_params["api_key"])
            nominatim_cache = {}
            prov_records    = []
            llm_calls = nom_calls = 0

            for idx, row in df_provincia.iterrows():
                # Use provincia_town as trigger word
                trigger_word = str(row.get("provincia_town", "") or "").strip()
                if not trigger_word:
                    trigger_word = str(row.get("Location", "") or "").lower().strip()

                centroid    = TOWN_CENTROIDS.get(trigger_word)
                real_addr   = has_real_address(row)
                cleaned     = None
                query       = None
                nom_result  = None
                accepted    = False
                flagged     = False
                tier        = None
                confidence  = None
                note        = None

                if real_addr:
                    cleaned    = llm_clean_address(row, trigger_word, llm_client)
                    llm_calls += 1
                    time.sleep(0.3)

                    query = (
                        f"{cleaned}, Italy"
                        if cleaned and "italy" not in cleaned.lower()
                        else (cleaned or f"{trigger_word}, Italy")
                    )

                    if query not in nominatim_cache:
                        nom_result = nominatim_geocode_full(query)
                        nominatim_cache[query] = nom_result
                        nom_calls += 1
                        time.sleep(1.1)
                    else:
                        nom_result = nominatim_cache[query]

                    accepted, flagged = nominatim_result_is_acceptable(nom_result)

                    if accepted and nom_result:
                        tier       = "5f" if flagged else "5"
                        confidence = "nominatim_provincia_flagged" if flagged else "nominatim_provincia"
                        note       = (
                            f"Nominatim provincia: {query} | "
                            f"type={nom_result['osm_type']} class={nom_result['osm_class']}"
                            + (" | FLAGGED" if flagged else "")
                        )
                    elif centroid:
                        tier       = "7b"
                        confidence = "centroid_nominatim_failed"
                        note       = f"Nominatim failed — centroid for {trigger_word}"
                        nom_result = None
                        accepted   = False
                    else:
                        tier       = None
                        confidence = "provincia_failed"
                        note       = f"Nominatim failed and no centroid for {trigger_word}"
                        nom_result = None
                        accepted   = False
                else:
                    if centroid:
                        tier       = "7a"
                        confidence = "centroid_direct"
                        note       = f"No address — centroid for {trigger_word}"
                    else:
                        tier       = None
                        confidence = "provincia_failed"
                        note       = f"No address and no centroid for {trigger_word}"

                prov_records.append({
                    "source_index":     idx,
                    "page_num":         row.get("page_num"),
                    "Name":             str(row.get("Name", "") or ""),
                    "Address":          str(row.get("Address", "") or ""),
                    "Location":         str(row.get("Location", "") or ""),
                    "section":          str(row.get("section", "") or ""),
                    "provincia_town":   trigger_word,
                    "has_real_address": real_addr,
                    "llm_cleaned":      cleaned,
                    "nominatim_query":  query,
                    "nom_lat":          nom_result["lat"]          if nom_result and accepted else None,
                    "nom_lon":          nom_result["lon"]          if nom_result and accepted else None,
                    "nom_type":         nom_result["osm_type"]     if nom_result else None,
                    "nom_class":        nom_result["osm_class"]    if nom_result else None,
                    "nom_display":      nom_result["display_name"] if nom_result else None,
                    "accepted":         accepted,
                    "flagged":          flagged,
                    "geocoding_tier":   tier,
                    "match_confidence": confidence,
                    "match_note":       note,
                })

                if (idx + 1) % 100 == 0:
                    print(f"  Progress: {idx+1}/{len(df_provincia)} | "
                          f"LLM={llm_calls} Nom={nom_calls}")

            provincia_decisions = pd.DataFrame(prov_records)
            print(f"\n  Decisions saved: {len(provincia_decisions)} rows")

        # ── Apply decisions ───────────────────────────────────
        print("\n  Applying provincia decisions ...")
        df_prov_result = df_provincia.copy().reset_index(drop=True)
        df_prov_result["geocoding_tier"]   = provincia_decisions["geocoding_tier"].values
        df_prov_result["match_confidence"] = provincia_decisions["match_confidence"].values
        df_prov_result["match_note"]       = provincia_decisions["match_note"].values
        df_prov_result["nominatim_query"]  = provincia_decisions["nominatim_query"].values
        df_prov_result["nominatim_flagged"] = provincia_decisions["flagged"].values

        def get_lat_prov(row_dec):
            if pd.notna(row_dec.get("nom_lat")) and row_dec.get("accepted"):
                return float(row_dec["nom_lat"])
            tw   = row_dec.get("provincia_town", "")
            tier = str(row_dec.get("geocoding_tier", ""))
            if tier in ("7a", "7b") and tw and tw in TOWN_CENTROIDS:
                return TOWN_CENTROIDS[tw][0]
            return None

        def get_lon_prov(row_dec):
            if pd.notna(row_dec.get("nom_lon")) and row_dec.get("accepted"):
                return float(row_dec["nom_lon"])
            tw   = row_dec.get("provincia_town", "")
            tier = str(row_dec.get("geocoding_tier", ""))
            if tier in ("7a", "7b") and tw and tw in TOWN_CENTROIDS:
                return TOWN_CENTROIDS[tw][1]
            return None

        df_prov_result["matched_lat"] = provincia_decisions.apply(get_lat_prov, axis=1)
        df_prov_result["matched_lon"] = provincia_decisions.apply(get_lon_prov, axis=1)

        _tier_prov = df_prov_result["geocoding_tier"].astype(str).str.replace(r'\.0$', '', regex=True)
        nom_provincia  = df_prov_result[_tier_prov.isin(["5", "5f"])].copy()
        cent_provincia = df_prov_result[_tier_prov.isin(["7a", "7b"])].copy()
        fail_provincia = df_prov_result[df_prov_result["matched_lat"].isna()].copy()

        print(f"  Tier 5  (Nominatim real address)   : {(_tier_prov == '5').sum()}")
        print(f"  Tier 5f (Nominatim flagged)         : {(_tier_prov == '5f').sum()}")
        print(f"  Tier 7a (centroid direct)           : {(_tier_prov == '7a').sum()}")
        print(f"  Tier 7b (centroid, Nominatim failed): {(_tier_prov == '7b').sum()}")
        print(f"  Failed  (no coordinates)            : {len(fail_provincia)}")

        # ── Save ──────────────────────────────────────────────
        _svc, _run_folder_id = _get_run_folder_id()

        if not nom_provincia.empty:
            save_or_upload_csv(nom_provincia, OUT_PROVINCIA_NOM_CSV, _svc, _run_folder_id)
            gdf_prov_nom = gpd.GeoDataFrame(
                nom_provincia,
                geometry=gpd.points_from_xy(
                    nom_provincia["matched_lon"].astype(float),
                    nom_provincia["matched_lat"].astype(float),
                ),
                crs="EPSG:4326",
            )
            save_or_upload_geojson(gdf_prov_nom, OUT_PROVINCIA_NOM_GEOJSON, _svc, _run_folder_id)
            print(f"\n  Nominatim provincia → {OUT_PROVINCIA_NOM_CSV.name} ({len(nom_provincia)} rows)")

        if not cent_provincia.empty:
            save_or_upload_csv(cent_provincia, OUT_PROVINCIA_CENTROID_CSV, _svc, _run_folder_id)
            gdf_prov_cent = gpd.GeoDataFrame(
                cent_provincia,
                geometry=gpd.points_from_xy(
                    cent_provincia["matched_lon"].astype(float),
                    cent_provincia["matched_lat"].astype(float),
                ),
                crs="EPSG:4326",
            )
            save_or_upload_geojson(gdf_prov_cent, OUT_PROVINCIA_CENTROID_GEOJSON, _svc, _run_folder_id)
            print(f"  Centroid provincia  → {OUT_PROVINCIA_CENTROID_CSV.name} ({len(cent_provincia)} rows)")

        if not fail_provincia.empty:
            save_or_upload_csv(fail_provincia, OUT_PROVINCIA_UNMATCHED, _svc, _run_folder_id)
            print(f"  Unmatched provincia → {OUT_PROVINCIA_UNMATCHED.name} ({len(fail_provincia)} rows)")

        # Decisions log
        save_or_upload_csv(provincia_decisions, OUT_DECISIONS_PROVINCIA, _svc, _run_folder_id)
        print(f"  Decisions log → {OUT_DECISIONS_PROVINCIA.name}")

        print(f"\n=== CELL 4B DONE ===")
        # Aliases for consistent naming
        matched_provincia  = nom_provincia
        centroid_provincia = cent_provincia
        unmatched_provincia = fail_provincia

else:
    print("\n  Cell 4B skipped (RUN_PROVINCIA=False or df_provincia empty)")
    nom_provincia  = pd.DataFrame()
    cent_provincia = pd.DataFrame()
    fail_provincia = pd.DataFrame()
    # Aliases for consistent naming
    matched_provincia  = nom_provincia
    centroid_provincia = cent_provincia
    unmatched_provincia = fail_provincia


=== CELL 4B — PROVINCIA GEOCODING ===
  Rows to process: 9132
  REUSE=False — fresh run: nominatim_20260722_0943

  Running LLM + Nominatim for provincia rows ...
  Progress: 100/9132 | LLM=0 Nom=0
  Progress: 200/9132 | LLM=1 Nom=1
  Progress: 300/9132 | LLM=1 Nom=1
  Progress: 400/9132 | LLM=1 Nom=1
  Progress: 500/9132 | LLM=1 Nom=1
  Progress: 600/9132 | LLM=1 Nom=1
  Progress: 700/9132 | LLM=2 Nom=2
  Progress: 800/9132 | LLM=9 Nom=7
  Progress: 900/9132 | LLM=11 Nom=8
  Progress: 1000/9132 | LLM=14 Nom=10
  Progress: 1100/9132 | LLM=14 Nom=10
  Progress: 1200/9132 | LLM=14 Nom=10
  Progress: 1300/9132 | LLM=14 Nom=10
  Progress: 1400/9132 | LLM=14 Nom=10
  Progress: 1500/9132 | LLM=28 Nom=21
  Progress: 1600/9132 | LLM=57 Nom=48
  Progress: 1700/9132 | LLM=107 Nom=92
  Progress: 1800/9132 | LLM=142 Nom=124
  Progress: 1900/9132 | LLM=213 Nom=192
  Progress: 2000/9132 | LLM=259 Nom=233
  Progress: 2100/9132 | LLM=332 Nom=292
  Progress: 2200/9132 | LLM=379 Nom=334
  Progress: 230

## Provincia town-name corrections


In [ ]:
# Assigns corrected centroids for provincia rows with OCR/propagation town-name mismatches.

# ── Step 1: Add new/alias entries to TOWN_CENTROIDS ──────────────────
TOWN_CENTROIDS_EXTRA = {
    # Key aliases — same town, different spelling from OCR
    "scorze'":          TOWN_CENTROIDS["scorzè"],
    "fiesso d'artico":  TOWN_CENTROIDS["fiesse d'artico"],
    "fosso’":           TOWN_CENTROIDS["fosso'"],
    "fossalta di piame": TOWN_CENTROIDS["fossalta di piave"],
    "chiarano":         TOWN_CENTROIDS["cinto caomaggiore"],
    "còna e pegollotte": TOWN_CENTROIDS["cona"],
    "pegollotte":       TOWN_CENTROIDS["cona"],
    "sottomarina":      TOWN_CENTROIDS["chioggia"],
    "stra":             (45.4108, 12.0378),
    "stra'":            (45.4108, 12.0378),
}

# ── Step 2: Page-level town reassignments ─────────────────────────────
PAGE_TOWN_OVERRIDE = {
    # venezia rows → correct town by page
    ("venezia", "550"): "portogruaro",
    ("venezia", "514"): "chioggia",
    ("venezia", "511"): "chioggia",
    ("venezia", "528"): "gruaro",
    ("venezia", "566"): "stra",
    # ad garbage → correct town by page
    ("maniaturate e filati", "524"): "fiesse d'artico",
    ("guida commerciale di venezia e provincia", "553"): "portogruaro",
    ("google", "570"): "stra",
    # other wrong propagation
    ("pegollotte", "517"): "cona",
    ("spolatore", "539"): "mirano",
    ("chiarano", "515"): "cinto caomaggiore",
    ("còna e pegollotte", "517"): "cona",
}

# Unresolvable — flag as out_of_scope
OUT_OF_SCOPE_TOWNS = {
    "padova",       # outside Venice province
    "nesto",      
    "zatti)",       
    "caravenealla", 
    "vigo",        
    "guida commerciale di venezia e provincia",  # almanac title 
    "google",       # ad garbage 
    "maniaturate e filati",  # product description 
}

# ── Load existing unmatched ───────────────────────────────────────────
_svc_p, _folder_p = _get_run_folder_id()
_pfiles = list_files_in_folder(_svc_p, _folder_p)

_prov_unm_f = next(f for f in _pfiles
                   if "unmatched_provincia" in f["name"]
                   and f["name"].endswith(".csv"))
_prov_cent_f = next(f for f in _pfiles
                    if "centroid_provincia_matched" in f["name"]
                    and f["name"].endswith(".csv"))

prov_unm  = pd.read_csv(io.StringIO(download_text(_svc_p, _prov_unm_f["id"])))
prov_cent = pd.read_csv(io.StringIO(download_text(_svc_p, _prov_cent_f["id"])))

print(f"Loaded unmatched: {len(prov_unm)} rows")
print(f"Loaded centroid matched: {len(prov_cent)} rows")

# ── Step 3: Apply fixes ───────────────────────────────────────────────
newly_matched = []
still_unmatched = []
out_of_scope_rows = []

MERGED_CENTROIDS = {**TOWN_CENTROIDS, **TOWN_CENTROIDS_EXTRA}

for _, row in prov_unm.iterrows():
    town     = str(row.get("provincia_town", "") or "").strip().lower()
    page_num = str(row.get("page_num", "") or "").strip()

    # Check page-level override first
    override_key = (town, page_num)
    if override_key in PAGE_TOWN_OVERRIDE:
        town = PAGE_TOWN_OVERRIDE[override_key]

    # Check out of scope
    if town in OUT_OF_SCOPE_TOWNS:
        r = row.copy()
        r["match_confidence"] = "out_of_scope_ad_propagation"
        r["match_note"]       = f"Ad/OCR propagation error: '{row.get('provincia_town','')}' is not a real town"
        r["geocoding_tier"]   = None
        out_of_scope_rows.append(r)
        continue

    # Try centroid lookup
    coords = MERGED_CENTROIDS.get(town)
    if coords and coords[0] is not None:
        r = row.copy()
        r["matched_lat"]      = coords[0]
        r["matched_lon"]      = coords[1]
        r["geocoding_tier"]   = "7a"
        r["match_confidence"] = "centroid_direct_patched"
        r["match_note"]       = (
            f"Patched: '{row.get('provincia_town','')}' → '{town}' centroid. "
            f"Original was OCR/propagation error."
            if town != row.get("provincia_town","").strip().lower()
            else f"Centroid for {town} (key alias fix)"
        )
        r["provincia_town"]   = town
        newly_matched.append(r)
    else:
        still_unmatched.append(row)

# ── Results ───────────────────────────────────────────────────────────
newly_matched_df   = pd.DataFrame(newly_matched)
still_unmatched_df = pd.DataFrame(still_unmatched)
out_of_scope_df    = pd.DataFrame(out_of_scope_rows)

print(f"\n=== PROVINCIA FIX RESULTS ===")
print(f"  Newly matched (centroid patched) : {len(newly_matched_df)}")
print(f"  Out-of-scope (ad propagation)    : {len(out_of_scope_df)}")
print(f"  Still unmatched                  : {len(still_unmatched_df)}")
print(f"  Total: {len(newly_matched_df)+len(out_of_scope_df)+len(still_unmatched_df)} "
      f"(was {len(prov_unm)})")
print()

if not newly_matched_df.empty:
    print("Patched town breakdown:")
    print(newly_matched_df["provincia_town"].value_counts().to_string())

if not out_of_scope_df.empty:
    print(f"\nOut-of-scope breakdown:")
    print(out_of_scope_df["provincia_town"].value_counts().to_string()
          if "provincia_town" in out_of_scope_df.columns
          else out_of_scope_df.get("match_note","").value_counts().to_string())

# ── Save updated files ────────────────────────────────────────────────
def _overwrite(file_id, name, df):
    content = df.to_csv(index=False)
    try:
        _local_path = RUN_DIR / name
        _local_path.parent.mkdir(parents=True, exist_ok=True)
        _local_path.write_text(content, encoding="utf-8")
    except OSError as e:
        print(f"  WARNING: local write failed for {name} — {e}")
    upload_text(_svc_p, content, name, _folder_p)
    print(f"  Saved (local+Drive): {name} ({len(df)} rows)")

# Updated centroid file = old centroid + newly matched
prov_cent_updated = pd.concat([prov_cent, newly_matched_df], ignore_index=True)
_overwrite(_prov_cent_f["id"], _prov_cent_f["name"], prov_cent_updated)

# Updated unmatched = still unmatched only (out_of_scope removed too)
_overwrite(_prov_unm_f["id"], _prov_unm_f["name"], still_unmatched_df)

# Save out-of-scope separately for stats
if not out_of_scope_df.empty:
    _svc_f2 = get_drive_service()
    _fin_f2  = FINAL_ID
    if _fin_f2:
        _oos_content = out_of_scope_df.to_csv(index=False)
        try:
            _oos_local_path = Path(FINAL_OUTPUT_DIR) / "provincia_out_of_scope.csv"
            _oos_local_path.parent.mkdir(parents=True, exist_ok=True)
            _oos_local_path.write_text(_oos_content, encoding="utf-8")
        except OSError as e:
            print(f"  WARNING: local write failed for provincia_out_of_scope.csv — {e}")
        upload_text(_svc_f2, _oos_content, "provincia_out_of_scope.csv", _fin_f2)
        print(f"  Saved (local+Drive): provincia_out_of_scope.csv ({len(out_of_scope_df)} rows)")


Loaded unmatched: 1 rows
Loaded centroid matched: 8711 rows

=== PROVINCIA FIX RESULTS ===
  Newly matched (centroid patched) : 0
  Out-of-scope (ad propagation)    : 0
  Still unmatched                  : 1
  Total: 1 (was 1)

  Saved (local+Drive): nominatim_20260722_0943_matched_centroid_provincia.csv (8711 rows)
  Saved (local+Drive): nominatim_20260722_0943_nominatim_unmatched_provincia.csv (1 rows)

Re-run stats to see updated counts.


## Inside-Venice Nominatim geocoding


In [15]:
#  CELL 5 — INSIDE VENICE NOMINATIM GEOCODING
#  RUN_INSIDE = True/False to enable/disable
#
#  Tiers:
#    6   — Nominatim matched named place (church, palazzo etc)
#    6f  — Nominatim matched, flagged/ambiguous
#
#  Outputs:
#    nominatim_geocoded_matched.csv/.geojson (appended)
#    nominatim_unmatched.csv (appended)
#    nominatim_decisions_inside.csv (audit log)
#
#  Skip condition: if decisions_inside.csv exists → reload

if RUN_INSIDE:
    print("\n=== CELL 5 — INSIDE VENICE NOMINATIM GEOCODING ===")

    REUSE_INSIDE_DECISIONS = False
    REAPPLY_AND_SAVE = True
    # REUSE_OUTSIDE_DECISIONS=True  + REAPPLY_AND_SAVE=False → fast load outputs into memory only
    # REUSE_OUTSIDE_DECISIONS=True  + REAPPLY_AND_SAVE=True  → re-apply decisions, re-save files
    # REUSE_OUTSIDE_DECISIONS=False + (any)                  → full fresh API run

    # ── Generate or reuse run name and output paths ──────────
    if REUSE_INSIDE_DECISIONS:
        _existing_dec = sorted(
            RUN_DIR.glob("*_nominatim_decisions_inside.csv"),
            key=lambda f: f.stat().st_mtime,
            reverse=True
        ) if RUN_DIR.exists() else []

        if _existing_dec:
            INSIDE_RUN_NAME = _existing_dec[0].name.replace(
                "_nominatim_decisions_inside.csv", ""
            )
            print(f"  REUSE_INSIDE_DECISIONS=True — reusing: {INSIDE_RUN_NAME}")
        else:
            _svc_scan, _scan_folder_id = _get_run_folder_id()
            _drive_decs = []
            if _svc_scan and _scan_folder_id:
                _drive_decs = [
                    f for f in list_files_in_folder(_svc_scan, _scan_folder_id)
                    if "_nominatim_decisions_inside.csv" in f["name"]
                ]
            if _drive_decs:
                _latest_dec = max(_drive_decs, key=lambda f: f["createdTime"])
                INSIDE_RUN_NAME = _latest_dec["name"].replace(
                    "_nominatim_decisions_inside.csv", ""
                )
                print(f"  REUSE_INSIDE_DECISIONS=True — reusing from Drive: {INSIDE_RUN_NAME}")
            else:
                print(f"  REUSE_INSIDE_DECISIONS=True but no decisions file found — running fresh")
                INSIDE_RUN_NAME = f"nominatim_{datetime.now().strftime('%Y%m%d_%H%M')}"
    else:
        INSIDE_RUN_NAME = f"nominatim_{datetime.now().strftime('%Y%m%d_%H%M')}"
        print(f"  REUSE_INSIDE_DECISIONS=False — new run: {INSIDE_RUN_NAME}")

    # ── Output paths ─────────────────────────────────────────
    OUT_INSIDE_NOM_CSV      = RUN_DIR / f"nominatim_inside_matched_{INSIDE_RUN_NAME}.csv"
    OUT_INSIDE_NOM_GEOJSON  = RUN_DIR / f"nominatim_inside_matched_{INSIDE_RUN_NAME}.geojson"
    OUT_INSIDE_UNMATCHED    = RUN_DIR / f"nominatim_inside_unmatched_{INSIDE_RUN_NAME}.csv"
    OUT_DECISIONS_INSIDE    = RUN_DIR / f"{INSIDE_RUN_NAME}_nominatim_decisions_inside.csv"
    OUT_STATS_MD            = RUN_DIR / f"{INSIDE_RUN_NAME}_nominatim_stats.md"


    # ── Fast load (kernel restore only) ─────────────────────
    if REUSE_INSIDE_DECISIONS and not REAPPLY_AND_SAVE:
        print(f"\n  REAPPLY_AND_SAVE=False — loading output files directly ...")
        _svc_load, _load_folder_id = _get_run_folder_id()
        nom_inside_df  = _load_output(OUT_INSIDE_NOM_CSV,   _svc_load, _load_folder_id)
        fail_inside_df = _load_output(OUT_INSIDE_UNMATCHED, _svc_load, _load_folder_id)
        print(f"  Nominatim inside  : {len(nom_inside_df)} rows")
        print(f"  Failed inside     : {len(fail_inside_df)} rows")
        print(f"\n=== CELL 5 DONE (fast load) ===")
    else:
        # ── Load Venice bbox ... ─────────────────────────────

        _sestiere_gdf = None
        if Path(SESTIERE_PATH).exists():
            _sestiere_gdf = gpd.read_file(SESTIERE_PATH)
            print(f"  Sestiere loaded locally: {len(_sestiere_gdf)} features")
        else:
            # Fallback: download from Drive
            _svc_bbox, _ = _get_run_folder_id()
            if _svc_bbox:
                _geo_folder_id = GEO_ID
                if _geo_folder_id:
                    _geo_files = list_files_in_folder(_svc_bbox, _geo_folder_id) \
                        if hasattr(_svc_bbox, 'files') else []
                    _sf = next((f for f in _geo_files if f["name"] == "sestiere.geojson"), None)
                    if _sf:
                        _raw = download_text(_svc_bbox, _sf["id"])
                        _tf = tempfile.NamedTemporaryFile(suffix=".geojson", delete=False, mode="w")
                        _tf.write(_raw); _tf.close()
                        _sestiere_gdf = gpd.read_file(_tf.name)
                        Path(_tf.name).unlink()
                        print(f"  Sestiere loaded from Drive: {len(_sestiere_gdf)} features")

        if _sestiere_gdf is None:
            raise RuntimeError(
                "Could not load sestiere.geojson — "
                "check SESTIERE_PATH in config.py or DRIVE_GEOSPATIAL_FOLDER on Drive"
            )

        _bbox = _sestiere_gdf.to_crs("EPSG:4326").total_bounds
        # total_bounds returns [min_lon, min_lat, max_lon, max_lat]
        VENICE_BBOX_MIN_LON = float(_bbox[0])
        VENICE_BBOX_MIN_LAT = float(_bbox[1])
        VENICE_BBOX_MAX_LON = float(_bbox[2])
        VENICE_BBOX_MAX_LAT = float(_bbox[3])

        print(f"  Venice bbox: lat {VENICE_BBOX_MIN_LAT:.4f}–{VENICE_BBOX_MAX_LAT:.4f}, "
            f"lon {VENICE_BBOX_MIN_LON:.4f}–{VENICE_BBOX_MAX_LON:.4f}")

        def point_in_venice_bbox(lat, lon):
            return (VENICE_BBOX_MIN_LAT <= lat <= VENICE_BBOX_MAX_LAT and
                    VENICE_BBOX_MIN_LON <= lon <= VENICE_BBOX_MAX_LON)


        #helpers

        def nominatim_geocode_full(query):
            # calls Nominatim with Venice viewbox bias, returns full result dict or None
            params = urllib.parse.urlencode({
                "q":            query,
                "format":       "json",
                "limit":        1,
                "countrycodes": "it",
                "addressdetails": 0,
                "viewbox": (f"{VENICE_BBOX_MIN_LON},{VENICE_BBOX_MIN_LAT},"
                            f"{VENICE_BBOX_MAX_LON},{VENICE_BBOX_MAX_LAT}"),
                # bounded=0 (default): biases toward bbox but allows results outside
                # if nothing is found inside — post-filter handles the rest
            })
            url = f"https://nominatim.openstreetmap.org/search?{params}"
            headers = {"User-Agent": "VeniceAlmanac1947/1.0 (thesis research)"}
            try:
                req = urllib.request.Request(url, headers=headers)
                with urllib.request.urlopen(req, timeout=10) as resp:
                    data = json.loads(resp.read().decode())
                if data:
                    hit = data[0]
                    return {
                        "lat":          float(hit["lat"]),
                        "lon":          float(hit["lon"]),
                        "osm_type":     hit.get("type", ""),
                        "osm_class":    hit.get("class", ""),
                        "display_name": hit.get("display_name", ""),
                    }
            except Exception:
                pass
            return None

        def nominatim_result_is_acceptable(result):
            # (accepted, flagged); rejects coords outside the Venice bbox, no centroid fallback for inside-Venice rows
            if result is None:
                return False, False

            # Post-filter: reject if coordinates fall outside historic Venice island
            lat = result.get("lat")
            lon = result.get("lon")
            if lat is not None and lon is not None:
                if not point_in_venice_bbox(lat, lon):
                    return False, False

            osm_type  = result.get("osm_type", "")
            osm_class = result.get("osm_class", "")
            if (osm_class, osm_type) in NOMINATIM_REJECT:
                return False, False
            if osm_type in NOMINATIM_ACCEPT_TYPES:
                return True, False
            if osm_class in NOMINATIM_ACCEPT_CLASSES:
                return True, False
            if osm_type in NOMINATIM_STREET_TYPES or osm_class == "highway":
                return True, True
            return True, True

        # ── Section-aware LLM query extractor ───────────────────
        def llm_extract_nominatim_queries_inside(rows_batch, client):
            # sends unmatched inside-Venice rows to GPT-4o with a section-aware prompt, returns {row, query, reasoning}
            rows_formatted = "\n".join([
                f"  Row {i+1}: Name={r['Name']} | "
                f"Address={r['Address']} | Location={r['Location']} | "
                f"Section={r.get('section', '')}"
                for i, r in enumerate(rows_batch)
            ])

            # Build section hints from the batch
            sections_in_batch = set(r.get("section", "") for r in rows_batch)
            section_context   = "\n".join(
                f"  - {s}: {SECTION_HINTS.get(s, 'use all available context')}"
                for s in sections_in_batch if s
            )

            prompt = f"""You are extracting geocodable address strings from rows of a 1947 Venetian commercial almanac.
    Each row failed to geocode using the Venetian civic number system (sestiere + number).
    These rows are classified as INSIDE VENICE — they refer to named places, institutions,
    churches, palazzi, or addresses within the historic Venice lagoon islands.

    Section-specific guidance for this batch:
    {section_context}

    Rules:
    - Always end the query with ", Venezia, Italy"
    - For churches: use full Italian name, e.g. "Chiesa di San Trovaso, Venezia, Italy" or Basilica : For the major Venetian basilicas, always use "Basilica" not "Chiesa": "Basilica di San Marco" 
    - If Address contains 'tit.' or 'titolo' followed by a church name, that IS a geocodable Venetian church. Expand it: 'tit. di S. Marco' → 'Basilica di San Marco, Venezia, Italy'. 'tit. SS. Giovanni e Paolo' → 'Basilica dei Santi Giovanni e Paolo, Venezia, Italy'.
    - For named palazzi, churches, campos and fondamente: omit the civic number
    - For Palazzo queries, use ONLY the palazzo name — never include the street, fondamenta, or civic number. 'Palazzo del Governo - fondamenta Zaguri 2662' → 'Palazzo del Governo, Venezia, Italy'.
    - For campos: "Campo [Name], Venezia, Italy"
    - For government institutions: full name, e.g. "Tribunale di Venezia, Venezia, Italy"
    - If Address and Location are both empty but Name contains a Venetian institution (Prefettura, Questura, Tribunale, Genio Civile, Intendenza, Capitaneria, Provveditorato, Ospedale, etc.), geocode the institution name directly: 'Prefettura di Venezia, Venezia, Italy'
    - For named buildings or fondamente: include sestiere if known
    - Include sestiere if present in Address or Location for disambiguation
    - Be as specific as possible — avoid vague queries like "San Marco, Venezia, Italy"
    - Expand Venetian abbreviations in queries: 'S.' → 'San/Santa', 'SS.' → 'Santi', 'F.TA' or 'FOND.' → 'Fondamenta', 'SOTTOPORT.' → 'Sottoportego', 'C.' → 'Calle', 'CAMP.' → 'Campo'. Example: 'SOTTOPORT. delle ACQUE 5013' → 'Sottoportego delle Acque, Venezia, Italy'
    - If nothing geocodable can be extracted, return null

    Here are the rows:
    {rows_formatted}

    Respond ONLY with a valid JSON array, no preamble, no markdown fences.
    Each element must have exactly these keys:
    {{"row": <1-based int>, "query": <string or null>, "reasoning": <one sentence>}}"""

            try:
                response = client.chat.completions.create(
                    model="gpt-4o",
                    max_tokens=1500,
                    temperature=0,
                    messages=[{"role": "user", "content": prompt}]
                )
                raw = response.choices[0].message.content.strip()
                if raw.startswith("```"):
                    raw = re.sub(r"^```[a-z]*\n?", "", raw)
                    raw = re.sub(r"\n?```$", "", raw)
                return json.loads(raw)
            except Exception as e:
                return [
                    {"row": i+1, "query": None,
                    "reasoning": f"LLM error: {e}"}
                    for i in range(len(rows_batch))
                ]

        # ── Skip condition ───────────────────────────────────────
        if OUT_DECISIONS_INSIDE.exists() or REUSE_INSIDE_DECISIONS:
            _loaded = False
            if OUT_DECISIONS_INSIDE.exists():
                try:
                    inside_decisions = pd.read_csv(OUT_DECISIONS_INSIDE)
                    print(f"\n  Decisions file found — loading {OUT_DECISIONS_INSIDE.name}")
                    print(f"  (no API calls) — {len(inside_decisions)} rows")
                    _loaded = True
                except OSError:
                    print(f"  Local read failed (Drive-mounted) — downloading via API ...")

            if not _loaded:
                _svc_tmp, _run_folder_id_tmp = _get_run_folder_id()
                _dec_files = _svc_tmp.files().list(
                    q=f"'{_run_folder_id_tmp}' in parents and trashed=false"
                    f" and name='{OUT_DECISIONS_INSIDE.name}'",
                    fields="files(id,name)",
                    supportsAllDrives=True, includeItemsFromAllDrives=True
                ).execute().get("files", [])
                if _dec_files:
                    inside_decisions = pd.read_csv(
                        io.StringIO(download_text(_svc_tmp, _dec_files[0]["id"]))
                    )
                    print(f"  Loaded from Drive: {len(inside_decisions)} rows")
                    _loaded = True

            if not _loaded:
                print(f"\n  No decisions file found — running LLM + Nominatim ...")
                _run_api = True
            else:
                _run_api = False

        else:
            print(f"\n  No decisions file — running LLM + Nominatim ...")
            _run_api = True

        if _run_api:
            with open(Path(OPENAI_KEY_FILE)) as f:
                _key_params = dict(
                    v.strip().split("=", 1) for v in f if "=" in v
                )
            llm_client      = openai.OpenAI(api_key=_key_params["api_key"])
            nominatim_cache = {}
            inside_records  = []
            BATCH_SIZE      = 10
            processed = llm_calls = nom_calls = 0

            inside_idx = list(range(len(df_inside)))

            for batch_start in range(0, len(inside_idx), BATCH_SIZE):
                batch_positions = inside_idx[batch_start:batch_start + BATCH_SIZE]
                batch_rows = [
                    {
                        "index":    pos,
                        "Name":     str(df_inside.iloc[pos].get("Name", "")     or ""),
                        "Address":  str(df_inside.iloc[pos].get("Address", "")  or ""),
                        "Location": str(df_inside.iloc[pos].get("Location", "") or ""),
                        "section":  str(df_inside.iloc[pos].get("section", "")  or ""),
                    }
                    for pos in batch_positions
                ]

                llm_results = llm_extract_nominatim_queries_inside(batch_rows, llm_client)
                llm_calls  += 1
                time.sleep(0.3)

                # Build lookup by 1-based row number — guarantees every row covered
                llm_lookup = {}
                for res in llm_results:
                    rn = res.get("row")
                    if rn is not None:
                        llm_lookup[int(rn)] = res

                # Process every row in batch — no row can be skipped
                for i, pos in enumerate(batch_positions):
                    row       = df_inside.iloc[pos]
                    res       = llm_lookup.get(i + 1, {})
                    query     = res.get("query")
                    reasoning = res.get("reasoning", "no result from LLM for this row")

                    nom_result = None
                    accepted   = False
                    flagged    = False

                    if query:
                        if query not in nominatim_cache:
                            nom_result = nominatim_geocode_full(query)
                            nominatim_cache[query] = nom_result
                            nom_calls += 1
                            time.sleep(1.1)
                        else:
                            nom_result = nominatim_cache[query]
                        accepted, flagged = nominatim_result_is_acceptable(nom_result)

                    inside_records.append({
                        "source_index":  pos,
                        "page_num":      row.get("page_num"),
                        "Name":          str(row.get("Name", "")     or ""),
                        "Address":       str(row.get("Address", "")  or ""),
                        "Location":      str(row.get("Location", "") or ""),
                        "section":       str(row.get("section", "")  or ""),
                        "llm_query":     query,
                        "llm_reasoning": reasoning,
                        "nom_lat":       nom_result["lat"]          if nom_result else None,
                        "nom_lon":       nom_result["lon"]          if nom_result else None,
                        "nom_type":      nom_result["osm_type"]     if nom_result else None,
                        "nom_class":     nom_result["osm_class"]    if nom_result else None,
                        "nom_display":   nom_result["display_name"] if nom_result else None,
                        "accepted":      accepted,
                        "flagged":       flagged,
                    })

                processed += len(batch_positions)
                if processed % 500 == 0 or processed == len(inside_idx):
                    print(f"  Progress: {processed}/{len(inside_idx)} | "
                        f"LLM={llm_calls} Nom={nom_calls}")

            inside_decisions = pd.DataFrame(inside_records)
            LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
            inside_decisions.to_csv(OUT_DECISIONS_INSIDE, index=False)
            print(f"\n  Decisions saved → {OUT_DECISIONS_INSIDE.name} "
                f"({len(inside_decisions)} rows)")

        # ── Apply inside decisions ───────────────────────────────
        print("\n  Applying inside-Venice decisions ...")

        df_inside_result = df_inside.copy()
        nom_inside_rows  = []
        fail_inside_rows = []
        nom_matched = nom_flagged = nom_failed = 0

        for _, res in inside_decisions.iterrows():
            pos      = int(res["source_index"])
            accepted = bool(res["accepted"]) if pd.notna(res["accepted"]) else False
            flagged  = bool(res["flagged"])  if pd.notna(res["flagged"])  else False
            query    = res["llm_query"] if pd.notna(res.get("llm_query")) else None

            row = df_inside.iloc[pos].to_dict()
            row["nominatim_query"]   = query
            row["nominatim_flagged"] = flagged

            if accepted and pd.notna(res.get("nom_lat")):
                row["matched_lat"]      = float(res["nom_lat"])
                row["matched_lon"]      = float(res["nom_lon"])
                row["match_confidence"] = "nominatim_venice_flagged" if flagged else "nominatim_venice"
                row["geocoding_tier"]   = "6f" if flagged else 6
                row["match_note"]       = (
                    f"Nominatim inside: {query} | "
                    f"type={res.get('nom_type')} class={res.get('nom_class')}"
                    + (" | FLAGGED" if flagged else "")
                )
                nom_inside_rows.append(row)
                nom_matched += 1
                if flagged:
                    nom_flagged += 1
            else:
                row["matched_lat"]      = None
                row["matched_lon"]      = None
                row["match_confidence"] = "nominatim_failed"
                row["geocoding_tier"]   = None
                row["match_note"]       = (
                    f"Nominatim failed: query={query} | "
                    f"type={res.get('nom_type')} class={res.get('nom_class')}"
                )
                fail_inside_rows.append(row)
                nom_failed += 1

        nom_inside_df   = pd.DataFrame(nom_inside_rows)
        fail_inside_df  = pd.DataFrame(fail_inside_rows)

        print(f"  Tier 6  (Nominatim inside Venice)  : {nom_matched - nom_flagged}")
        print(f"  Tier 6f (Nominatim inside, flagged) : {nom_flagged}")
        print(f"  Failed  (Nominatim could not resolve): {nom_failed}")

        # ── Save inside-Venice outputs ───────────────────────────
        _svc, _run_folder_id = _get_run_folder_id()


        if not nom_inside_df.empty:
            save_or_upload_csv(nom_inside_df, OUT_INSIDE_NOM_CSV, _svc, _run_folder_id)
            gdf_nom_in = gpd.GeoDataFrame(
                nom_inside_df,
                geometry=gpd.points_from_xy(
                    nom_inside_df["matched_lon"].astype(float),
                    nom_inside_df["matched_lat"].astype(float),
                ),
                crs="EPSG:4326",
            )
            save_or_upload_geojson(gdf_nom_in, OUT_INSIDE_NOM_GEOJSON, _svc, _run_folder_id)
            print(f"\n  Nominatim geocoded (inside) → {OUT_INSIDE_NOM_CSV.name} ({len(nom_inside_df)} rows)")

        if not fail_inside_df.empty:
            save_or_upload_csv(fail_inside_df, OUT_INSIDE_UNMATCHED, _svc, _run_folder_id)
            print(f"  Inside unmatched → {OUT_INSIDE_UNMATCHED.name} ({len(fail_inside_df)} rows)")

        # Decisions log
        # Decisions log — upload to Drive only (local path is Drive-mounted, write would fail)
        if _svc and _run_folder_id:
            try:
                upload_text(_svc, inside_decisions.to_csv(index=False),
                            OUT_DECISIONS_INSIDE.name, _run_folder_id)
                print(f"  Decisions log → {OUT_DECISIONS_INSIDE.name} (Drive only)")
            except Exception as e:
                print(f"  WARNING: could not save decisions log — {e}")

        print(f"\n=== CELL 5 DONE ===")

        # Aliases for consistent naming across notebook
        matched_inside   = nom_inside_df
        unmatched_inside = fail_inside_df

else:
    print("\n  Cell 5 skipped (RUN_INSIDE=False)")
    nom_inside_df  = pd.DataFrame()
    fail_inside_df = pd.DataFrame()
    # Aliases for consistent naming across notebook
    matched_inside   = nom_inside_df
    unmatched_inside = fail_inside_df





=== CELL 5 — INSIDE VENICE NOMINATIM GEOCODING ===
  REUSE_INSIDE_DECISIONS=False — new run: nominatim_20260722_1042
  Sestiere loaded locally: 8 features
  Venice bbox: lat 45.4231–45.4493, lon 12.3037–12.3671

  No decisions file — running LLM + Nominatim ...
  Progress: 500/4816 | LLM=50 Nom=28
  Progress: 1000/4816 | LLM=100 Nom=62
  Progress: 1500/4816 | LLM=150 Nom=114
  Progress: 2000/4816 | LLM=200 Nom=244
  Progress: 2500/4816 | LLM=250 Nom=360
  Progress: 3000/4816 | LLM=300 Nom=455
  Progress: 3500/4816 | LLM=350 Nom=513
  Progress: 4000/4816 | LLM=400 Nom=567
  Progress: 4500/4816 | LLM=450 Nom=619
  Progress: 4816/4816 | LLM=482 Nom=649

  Decisions saved → nominatim_20260722_1042_nominatim_decisions_inside.csv (4816 rows)

  Applying inside-Venice decisions ...
  Tier 6  (Nominatim inside Venice)  : 912
  Tier 6f (Nominatim inside, flagged) : 506
  Failed  (Nominatim could not resolve): 3398

  Nominatim geocoded (inside) → nominatim_20260722_1042_matched_nominatim_insid

## Pages 36-37 church-name fix


In [21]:
# Extracts church name from a 'tit.' prefix into Address for pages 36-37; clears tit. from Role.
# Updates clean_pages and the nb06 matched/unmatched inside-Venice files for these pages.

def extract_church_name(val):
    # 'tit. di S. Marco' -> 'S. Marco'
    if pd.isna(val):
        return None
    s = re.sub(
        r'^tit\.\s*(di\s+|dei\s+|della\s+|dello\s+|dell\'\s*|de\s+)?',
        '', str(val).strip(), flags=re.IGNORECASE
    ).strip()
    return s if s else None

def fix_tit(df, filter_pages=None):
    # applies the tit. fix to a dataframe, optionally filtered to filter_pages
    df = df.copy()
    if filter_pages and "page_num" in df.columns:
        page_filter = df["page_num"].astype(str).isin(filter_pages)
    else:
        page_filter = pd.Series([True] * len(df), index=df.index)

    # Fix tit. in Address
    if "Address" in df.columns:
        addr_mask = page_filter & df["Address"].fillna("").str.contains(
            r"tit\.", case=False, regex=True)
        df.loc[addr_mask, "Address"] = df.loc[addr_mask, "Address"].apply(
            extract_church_name)

    # Fix tit. in Role → move to Address, clear Role
    if "Role" in df.columns:
        role_mask = page_filter & df["Role"].fillna("").str.contains(
            r"tit\.", case=False, regex=True)
        df.loc[role_mask, "Address"] = df.loc[role_mask, "Role"].apply(
            extract_church_name)
        df.loc[role_mask, "Role"] = None

    return df

# ── 1. Fix nb06 unmatched inside ─────────────────────────────────────
print("Fixing nb06 unmatched inside ...")
_svc, _folder_id = _get_run_folder_id()
_files = list_files_in_folder(_svc, _folder_id)

_unm_f = next(f for f in _files if "unmatched_inside" in f["name"]
              and f["name"].endswith(".csv"))
df_unm = pd.read_csv(io.StringIO(download_text(_svc, _unm_f["id"])))
df_unm_fixed = fix_tit(df_unm, filter_pages=["36","37"])
changed = (df_unm["Address"].fillna("") != df_unm_fixed["Address"].fillna("")).sum()
print(f"  Rows changed: {changed}")
save_or_upload_csv(df_unm_fixed, RUN_DIR / _unm_f["name"], _svc, _folder_id)

# ── 2. Fix nb06 matched inside ────────────────────────────────────────
print("\nFixing nb06 matched inside ...")
_mat_f = next(f for f in _files if "nominatim_inside_matched" in f["name"]
              and f["name"].endswith(".csv"))
df_mat = pd.read_csv(io.StringIO(download_text(_svc, _mat_f["id"])))
df_mat_fixed = fix_tit(df_mat, filter_pages=["36","37"])
changed = (df_mat["Address"].fillna("") != df_mat_fixed["Address"].fillna("")).sum()
print(f"  Rows changed: {changed}")
save_or_upload_csv(df_mat_fixed, RUN_DIR / _mat_f["name"], _svc, _folder_id)

# ── 3. Fix clean_pages 36 and 37 ─────────────────────────────────────
print("\nFixing clean_pages ...")
_svc_c  = get_drive_service()
_cp_c   = CLEAN_PAGES_ID

for pg in ["36", "37"]:
    _pf  = find_folder(_svc_c, f"page_{pg}", _cp_c)
    _ff  = list_files_in_folder(_svc_c, _pf)
    _sem = next(f for f in _ff if f["name"].endswith("_semantic.csv"))
    _df  = pd.read_csv(io.StringIO(download_text(_svc_c, _sem["id"])))
    _df_fixed = fix_tit(_df)
    changed = (_df["Address"].fillna("") != _df_fixed["Address"].fillna("")).sum()
    print(f"  page_{pg}: {changed} rows changed")
    save_or_upload_csv(_df_fixed, Path(CLEAN_PAGES_DIR) / f"page_{pg}" / _sem["name"], _svc_c, _pf)

print("\n=== tit. fix done ✓ ===")
print("Files updated:")
print("  - nb06 unmatched inside CSV")
print("  - nb06 matched inside CSV")
print("  - clean_pages/page_36 semantic CSV")
print("  - clean_pages/page_37 semantic CSV")


Fixing nb06 unmatched inside ...
  Rows changed: 0
  Saved: nominatim_20260722_1042_nominatim_unmatched_inside.csv

Fixing nb06 matched inside ...
  Rows changed: 0
  Saved: nominatim_20260722_1042_matched_nominatim_inside.csv

Fixing clean_pages ...
  page_36: 0 rows changed
  Saved: page_36_semantic.csv
  page_37: 0 rows changed
  Saved: page_37_semantic.csv

=== tit. fix done ✓ ===
Files updated:
  - nb06 unmatched inside CSV
  - nb06 matched inside CSV
  - clean_pages/page_36 semantic CSV
  - clean_pages/page_37 semantic CSV
